In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys

In [ ]:
import json
import pandas as pd
import requests
from tqdm import tqdm
import time
import duckdb

from dotenv import load_dotenv
load_dotenv()
REPO = os.getenv("REPO_PATH")

In [6]:
fp_inat = '/Users/tplas/data/2026-06-29 iNaturalist GBIF/0005214-260623161305970.csv'

with open(fp_inat) as f:
    for i, line in zip(range(5), f):
        print(line)

gbifID	datasetKey	occurrenceID	kingdom	phylum	class	order	family	genus	species	infraspecificEpithet	taxonRank	scientificName	verbatimScientificName	verbatimScientificNameAuthorship	countryCode	locality	stateProvince	occurrenceStatus	individualCount	publishingOrgKey	decimalLatitude	decimalLongitude	coordinateUncertaintyInMeters	coordinatePrecision	elevation	elevationAccuracy	depth	depthAccuracy	eventDate	day	month	year	taxonKey	speciesKey	basisOfRecord	institutionCode	collectionCode	catalogNumber	recordNumber	identifiedBy	dateIdentified	license	rightsHolder	recordedBy	typeStatus	establishmentMeans	lastInterpreted	mediaType	issue

3079864577	50c9509d-22c7-4a22-a47d-8c48425ef4a7	https://www.inaturalist.org/observations/72892375	Plantae	Tracheophyta	Liliopsida	Asparagales	Iridaceae	Iris	Iris attica		SPECIES	Iris attica Boiss. & Heldr.	Iris attica		GR		Attica	PRESENT		28eb1a3f-1c15-4a95-931a-4af90ecb574d	38.203363	23.761756							2021-04-04T12:27:11	4	4	2021	7WR7Z	7WR7Z	HUMAN_OBSERVATION	iN

In [7]:

result = duckdb.sql("""
    SELECT species, COUNT(*) as cnt
    FROM read_csv_auto('/Users/tplas/data/2026-06-29 iNaturalist GBIF/0005214-260623161305970.csv', types={'species': 'VARCHAR'})
    GROUP BY species
    ORDER BY cnt DESC
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
n_species = 100 
target_n_obs = 1000 

result['diff_target'] = abs(result['cnt'] - target_n_obs)
target_species = result.nsmallest(n_species, 'diff_target')['species'].tolist()
print(len(target_species))
for s in target_species:
    print(s, result[result['species'] == s].cnt)

100
Sphenophorus interstitialis 19181    1000
Name: cnt, dtype: int64
Hymeniacidon perlevis 19182    1000
Name: cnt, dtype: int64
Senega cruciata 19183    1000
Name: cnt, dtype: int64
Rusa timorensis 19184    1000
Name: cnt, dtype: int64
Lactuca graminifolia 19185    1000
Name: cnt, dtype: int64
Lomandra obliqua 19186    1000
Name: cnt, dtype: int64
Erica canariensis 19187    1000
Name: cnt, dtype: int64
Luridiblatta trivittata 19188    1000
Name: cnt, dtype: int64
Phebalium squamulosum 19189    1000
Name: cnt, dtype: int64
Spragueia dama 19190    1000
Name: cnt, dtype: int64
Telicota colon 19191    1000
Name: cnt, dtype: int64
Rhinomuraena quaesita 19192    1000
Name: cnt, dtype: int64
Idea leuconoe 19193    1000
Name: cnt, dtype: int64
Euplectes progne 19165    1001
Name: cnt, dtype: int64
Dioctria linearis 19166    1001
Name: cnt, dtype: int64
Keckiella breviflora 19167    1001
Name: cnt, dtype: int64
Pycnoscelus indicus 19168    1001
Name: cnt, dtype: int64
Lepomis marginatus 19169

In [17]:
target_list = ", ".join(f"'{t}'" for t in target_species)

filtered = duckdb.sql(f"""
    SELECT * FROM read_csv_auto('{fp_inat}', types={{'species': 'VARCHAR'}})
    WHERE species IN ({target_list})
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:
filtered.license.value_counts()

license
CC_BY_NC_4_0    79601
CC_BY_4_0       15577
CC0_1_0          4840
Name: count, dtype: int64

In [ ]:
folder_output = os.path.join(REPO, 'outputs/inaturalist')
folder_sel_name = f"inaturalist_{len(target_species)}_species_{target_n_obs}_obs"
os.makedirs(os.path.join(folder_output, folder_sel_name), exist_ok=True)

for t in target_species:
    df_sel = filtered[filtered['species'] == t]
    # print(t, len(df_sel))
    name_file = f"{t.replace(' ', '_').lower()}.csv"
    df_sel.reset_index(drop=True, inplace=True)
    df_sel.to_csv(os.path.join(folder_output, folder_sel_name, name_file), index=False)